In [ ]:
import json
import zstandard as zstd
from datasets import Dataset
import re
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

!rm -f saiga_dataset.jsonl.zst
!wget -O saiga_dataset.jsonl.zst https://huggingface.co/datasets/IlyaGusev/ru_turbo_saiga/resolve/main/ru_turbo_saiga.jsonl.zst

with open("saiga_dataset.jsonl.zst", "rb") as f:
    dctx = zstd.ZstdDecompressor()
    text_data = dctx.decompress(f.read()).decode("utf-8")

raw_saiga = [json.loads(line) for line in text_data.strip().split("\n") if line]

def is_friendly(text):
    if any(kw in text.lower() for kw in ["код", "python", "linux", "ошибка", "ip", "порт", "html", "команда"]):
        return False
    if "http" in text or "```" in text:
        return False
    if len(text.split()) > 70 or len(text.split()) < 3:
        return False
    return True

saiga_pairs = []
for dialog in raw_saiga:
    messages = dialog.get("messages", [])
    for i in range(0, len(messages) - 1, 2):
        user_msg = messages[i]
        bot_msg = messages[i+1]
        if user_msg["role"] == "user" and bot_msg["role"] == "bot":
            if is_friendly(bot_msg["content"]):
                saiga_pairs.append({
                    "instruction": user_msg["content"].strip(),
                    "output": bot_msg["content"].strip()
                })
        if len(saiga_pairs) >= 1000:
            break
    if len(saiga_pairs) >= 1000:
        break

print(f"Отобрано {len(saiga_pairs)} 'мягких' примеров из Saiga")
path_to_my_rules = "/content/drive/MyDrive/code.json"

with open(path_to_my_rules, 'r', encoding='utf-8') as f:
    charming_data = json.load(f)

print(f"Загружено {len(charming_data)} примеров")


combined = charming_data + saiga_pairs

with open("combined_dataset.jsonl", "w", encoding="utf-8") as f:
    for entry in combined:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

!cp combined_dataset.jsonl /content/drive/MyDrive/
